# 09 — M5: YOLOv8n-cls · Kaggle GPU Training
**Driver Drowsiness Detection** — 5-fold subject-independent CV on UL-DD infrared frames.

### Setup
1. Upload `yolo_frames/` as a **Kaggle Dataset** (New Dataset → upload the folder)
2. Add that dataset to this notebook via **Add Data** (right panel)
3. Set **Accelerator → GPU T4 x2** (Settings panel)
4. Run All


In [ ]:
# ── Step 1: Check what Kaggle already has ─────────────────────────────────────
import subprocess, sys, os, functools

# Disable wandb prompt and Ray Tune callback
os.environ['WANDB_MODE'] = 'disabled'
os.environ['WANDB_DISABLED'] = 'true'
os.environ['RAY_TUNE_DISABLE'] = '1'

import torch
print(f'PyTorch  : {torch.__version__}')

try:
    import ultralytics
    print(f'ultralytics: {ultralytics.__version__} (pre-installed)')
    NEED_INSTALL = False
except ImportError:
    print('ultralytics: NOT installed — will install')
    NEED_INSTALL = True

# ── Step 2: Patch torch.load ONCE for PyTorch >= 2.6 ─────────────────────────
if not getattr(torch.load, '_patched', False):
    _real_load = torch.load
    def _safe_load(*a, **kw):
        kw.setdefault('weights_only', False)
        return _real_load(*a, **kw)
    _safe_load._patched = True
    torch.load = _safe_load
    print('Patched torch.load')

# ── Step 3: Install ultralytics only if needed ───────────────────────────────
if NEED_INSTALL:
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install',
                           'ultralytics==8.2.50', '--no-deps'])
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install',
                           'ultralytics==8.2.50'])
    print('Installed ultralytics==8.2.50')

# ── Step 3b: Neutralise broken Ray Tune callback ─────────────────────────────
try:
    from ultralytics.utils.callbacks import raytune
    raytune.callbacks = {}  # remove all ray tune callbacks
    print('Disabled Ray Tune callbacks')
except Exception:
    pass

# ── Step 4: GPU info ─────────────────────────────────────────────────────────
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram = props.total_memory / 1e9
    print(f'GPU      : {props.name}  ({vram:.1f} GB)')
    BATCH = 64
else:
    print('⚠ No GPU — will be slow!')
    BATCH = 32
print(f'Batch    : {BATCH}')

PyTorch  : 2.10.0+cpu
ultralytics: 8.4.23 (pre-installed)
Patched torch.load
CUDA     : False
⚠ No GPU — will be slow!
Batch    : 32


In [ ]:
import os, shutil, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

from ultralytics import YOLO

# ── Paths (Kaggle) ────────────────────────────────────────────────────────────
# Your uploaded dataset appears under /kaggle/input/<dataset-name>/
# Adjust DATASET_NAME to match what you named it on Kaggle
DATASET_NAME = 'yolo-frames'   # ← change if you named it differently
INPUT_DIR    = Path(f'/kaggle/input/{DATASET_NAME}')
WORK_DIR     = Path('/kaggle/working')
YOLO_DIR     = WORK_DIR / 'yolo_cls'
CKPT_DIR     = WORK_DIR / 'checkpoints'
REPORT_DIR   = WORK_DIR / 'reports'
for d in [YOLO_DIR, CKPT_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Auto-detect frame directory ───────────────────────────────────────────────
# Handle cases where Kaggle nests: /kaggle/input/yolo-frames/yolo_frames/
candidates = [
    INPUT_DIR,
    INPUT_DIR / 'yolo_frames',
    INPUT_DIR / 'yolo-frames',
]
FRAME_DIR = None
for c in candidates:
    if (c / 'metadata.csv').exists():
        FRAME_DIR = c
        break
if FRAME_DIR is None:
    print('Could not find metadata.csv. Contents of INPUT_DIR:')
    for p in sorted(INPUT_DIR.rglob('*'))[:30]:
        print(f'  {p}')
    raise FileNotFoundError('Upload yolo_frames/ with metadata.csv as a Kaggle Dataset')

print(f'FRAME_DIR : {FRAME_DIR}')
print(f'YOLO_DIR  : {YOLO_DIR}')
print(f'CKPT_DIR  : {CKPT_DIR}')

# ── Constants ─────────────────────────────────────────────────────────────────
SUBJECTS    = list('ABCDEFGHIJKLMNOPQRS')
NO_VIDEO    = {'B', 'I', 'M'}
CLASS_NAMES = ['Alert', 'LowVigilant', 'Drowsy']
N_CLASSES   = 3

FOLDS = [
    list('ABCD'),
    list('EFGH'),
    list('IJK'),
    list('LMNO'),
    list('PQRS'),
]

MODEL_NAME  = 'M5'
YOLO_EPOCHS = 20
YOLO_IMGSZ  = 224
YOLO_BASE   = 'yolov8n-cls.pt'

print('Ready.')


In [ ]:
# ── Fix Windows paths in metadata.csv ─────────────────────────────────────────
meta_df = pd.read_csv(FRAME_DIR / 'metadata.csv')
print(f'Total frames in metadata: {len(meta_df):,}')

# Remap Windows absolute paths → Kaggle paths
sample = meta_df['path'].iloc[0]
print(f'Sample path (original): {sample}')

if '\\' in sample or 'C:' in sample.upper():
    import re
    def remap(p):
        m = re.search(r'yolo_frames[/\\](.+)', p.replace('\\', '/'))
        return str(FRAME_DIR / m.group(1)) if m else p
    meta_df['path'] = meta_df['path'].apply(remap)
    print(f'Remapped to Kaggle paths')
    print(f'Sample path (fixed): {meta_df["path"].iloc[0]}')

# Verify a few files exist
exists = meta_df['path'].head(20).apply(lambda p: Path(p).exists())
print(f'First 20 files exist: {exists.sum()}/20')


In [ ]:
# ── Dataset overview ───────────────────────────────────────────────────────────
print(f'Subjects: {sorted(meta_df["subject"].unique())}')
cls_counts = meta_df['class_name'].value_counts().reindex(CLASS_NAMES)
print()
for cn, cnt in cls_counts.items():
    print(f'  {cn:15s}: {cnt:6,d}  ({cnt/len(meta_df)*100:.1f}%)')

fig, ax = plt.subplots(figsize=(6, 4))
colors = ['#2ecc71', '#f39c12', '#e74c3c']
ax.bar(CLASS_NAMES, cls_counts.values, color=colors, edgecolor='white')
ax.set_title('Overall Class Distribution')
ax.set_ylabel('Frame count')
plt.tight_layout()
plt.show()


In [ ]:
# ── Build YOLO fold directories (symlinks — no extra disk) ────────────────────
t0 = time.time()

for fold_idx, test_subjects in enumerate(FOLDS):
    fold_dir = YOLO_DIR / f'fold_{fold_idx}'
    test_subs  = [s for s in test_subjects if s not in NO_VIDEO]
    train_subs = [s for s in SUBJECTS if s not in NO_VIDEO and s not in test_subjects]

    test_df  = meta_df[meta_df['subject'].isin(test_subs)]
    train_df = meta_df[meta_df['subject'].isin(train_subs)]

    # Check if already built
    n_existing = sum(1 for _ in (fold_dir / 'train').rglob('*.jpg')) if (fold_dir / 'train').exists() else 0
    if n_existing > 100:
        print(f'Fold {fold_idx}: already built ({n_existing} train files) — skipping build')
    else:
        for split in ['train', 'val']:
            for cn in CLASS_NAMES:
                (fold_dir / split / cn).mkdir(parents=True, exist_ok=True)

        def _link(df, split):
            n = 0
            for _, row in df.iterrows():
                src = Path(row['path'])
                if not src.exists():
                    continue
                dst = fold_dir / split / row['class_name'] / f"{row['subject']}_{src.name}"
                if not dst.exists():
                    try:
                        os.symlink(str(src), str(dst))
                    except OSError:
                        shutil.copy2(str(src), str(dst))
                n += 1
            return n

        n_tr = _link(train_df, 'train')
        n_te = _link(test_df,  'val')
        print(f'Fold {fold_idx}: {n_tr} train + {n_te} val  |  test={test_subs}')

    # ALWAYS remove empty class dirs (YOLO crashes on them)
    for split in ['train', 'val']:
        for cn in CLASS_NAMES:
            d = fold_dir / split / cn
            if d.exists() and not any(d.iterdir()):
                d.rmdir()
                print(f'  ⚠ Removed empty {split}/{cn} for fold {fold_idx}')

print(f'Done in {time.time()-t0:.0f}s')

In [ ]:
# ── DELETE old (broken) checkpoints so training reruns ─────────────────────────
# Run this ONCE, then you can delete/skip this cell
import shutil
ckpt_path = Path('/kaggle/working/checkpoints')
if ckpt_path.exists():
    shutil.rmtree(ckpt_path)
    print(f'✓ Deleted old checkpoints at {ckpt_path}')
else:
    print('No old checkpoints found — ready to train fresh')

In [ ]:
# ── Train 5 folds with GPU ─────────────────────────────────────────────────────
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

all_fold_results = []
device = 0 if torch.cuda.is_available() else 'cpu'

# YOLO doesn't allow absolute paths in 'project', so we work from WORK_DIR
os.chdir(str(WORK_DIR))

for fold_idx, test_subjects in enumerate(FOLDS):
    print(f'\n{"="*60}')
    print(f'  Fold {fold_idx}  |  test = {[s for s in test_subjects if s not in NO_VIDEO]}')
    print(f'{"="*60}')

    fold_dir = YOLO_DIR / f'fold_{fold_idx}'
    ckpt_rel = 'checkpoints'  # relative to WORK_DIR
    best_pt  = WORK_DIR / ckpt_rel / f'M5_fold{fold_idx}' / 'weights' / 'best.pt'

    test_subs  = [s for s in test_subjects if s not in NO_VIDEO]
    train_subs = [s for s in SUBJECTS if s not in NO_VIDEO and s not in test_subjects]
    test_df  = meta_df[meta_df['subject'].isin(test_subs)]
    train_df = meta_df[meta_df['subject'].isin(train_subs)]

    # ── Train or skip ─────────────────────────────────────────────────────
    if best_pt.exists():
        print(f'  ✓ best.pt exists — SKIPPING training')
        model = YOLO(str(best_pt))
    else:
        print(f'  Training on {device} (epochs={YOLO_EPOCHS}, batch={BATCH}) ...')
        model = YOLO(YOLO_BASE)
        model.train(
            data      = str(fold_dir),
            epochs    = YOLO_EPOCHS,
            imgsz     = YOLO_IMGSZ,
            batch     = BATCH,
            device    = device,
            optimizer = 'SGD',
            project   = ckpt_rel,
            name      = f'M5_fold{fold_idx}',
            exist_ok  = True,
            verbose   = True,
            patience  = 15,
            lr0       = 1e-3,
            lrf       = 0.01,
            dropout   = 0.3,
        )
        if best_pt.exists():
            model = YOLO(str(best_pt))
            print(f'  ✓ Loaded best.pt')

    # ── Evaluate with correct class-index mapping ─────────────────────────
    yolo_names = model.names
    yolo_to_ours = {}
    for yolo_idx, yolo_name in yolo_names.items():
        yolo_to_ours[yolo_idx] = CLASS_NAMES.index(yolo_name)
    print(f'  Class mapping: {yolo_names}')
    print(f'  yolo→ours: {yolo_to_ours}')

    y_true, y_pred = [], []
    for cn_idx, cn in enumerate(CLASS_NAMES):
        class_dir = fold_dir / 'val' / cn
        if not class_dir.exists():
            continue
        imgs = sorted(class_dir.glob('*.jpg'))
        if not imgs:
            continue
        preds = model.predict(source=str(class_dir), verbose=False, batch=BATCH)
        for p in preds:
            y_true.append(cn_idx)
            y_pred.append(yolo_to_ours[int(p.probs.top1)])

    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    print(f'  ► Fold {fold_idx}: acc={acc:.4f}  f1={f1:.4f}')

    all_fold_results.append({
        'fold': fold_idx, 'accuracy': acc, 'macro_f1': f1,
        'y_true': y_true, 'y_pred': y_pred,
        'test_subjects': test_subs,
        'n_train': len(train_df), 'n_test': len(test_df),
    })

# ── Summary ───────────────────────────────────────────────────────────────────
accs = [f['accuracy'] for f in all_fold_results]
f1s  = [f['macro_f1'] for f in all_fold_results]
print(f'\n{"="*60}')
print(f'M5 Mean Accuracy : {np.mean(accs):.4f} ± {np.std(accs):.4f}')
print(f'M5 Mean Macro-F1 : {np.mean(f1s):.4f} ± {np.std(f1s):.4f}')
print(f'{"="*60}')

In [ ]:
# ── Visualizations ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

folds_x = [f'Fold {f["fold"]}' for f in all_fold_results]
accs = [f['accuracy'] for f in all_fold_results]
f1s  = [f['macro_f1'] for f in all_fold_results]

axes[0].bar(folds_x, accs, color='steelblue', edgecolor='white')
axes[0].axhline(np.mean(accs), color='red', ls='--', label=f'Mean={np.mean(accs):.3f}')
axes[0].set_title('M5 — Per-fold Accuracy')
axes[0].set_ylim(0, 1); axes[0].legend()

axes[1].bar(folds_x, f1s, color='darkorange', edgecolor='white')
axes[1].axhline(np.mean(f1s), color='red', ls='--', label=f'Mean={np.mean(f1s):.3f}')
axes[1].set_title('M5 — Per-fold Macro-F1')
axes[1].set_ylim(0, 1); axes[1].legend()

plt.tight_layout()
plt.savefig(REPORT_DIR / 'M5_fold_results.png', dpi=150)
plt.show()

# Confusion matrix
all_true, all_pred = [], []
for f in all_fold_results:
    all_true.extend(f['y_true']); all_pred.extend(f['y_pred'])

cm = confusion_matrix(all_true, all_pred, labels=[0,1,2])
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('M5 — Confusion Matrix (all folds)')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'M5_confusion_matrix.png', dpi=150)
plt.show()


In [ ]:
# ── Results table ──────────────────────────────────────────────────────────────
rows = []
for f in all_fold_results:
    rows.append({
        'Fold': f['fold'],
        'Test Subjects': ', '.join(f['test_subjects']),
        'Train': f['n_train'], 'Test': f['n_test'],
        'Accuracy': f['accuracy'], 'Macro-F1': f['macro_f1'],
    })
rdf = pd.DataFrame(rows)
rdf.loc['Mean'] = ['','', rdf['Train'].mean(), rdf['Test'].mean(),
                    rdf['Accuracy'].mean(), rdf['Macro-F1'].mean()]
rdf.loc['Std']  = ['','','','', rdf['Accuracy'].std(), rdf['Macro-F1'].std()]
rdf.style.format({'Accuracy':'{:.4f}','Macro-F1':'{:.4f}'})


In [ ]:
# ── Save results ──────────────────────────────────────────────────────────────
def _ser(obj):
    if isinstance(obj, (np.integer,)):  return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, np.ndarray):     return obj.tolist()
    if isinstance(obj, dict):           return {k: _ser(v) for k, v in obj.items()}
    if isinstance(obj, list):           return [_ser(i) for i in obj]
    return obj

results = {
    'model_name': MODEL_NAME,
    'mean_acc': float(np.mean([f['accuracy'] for f in all_fold_results])),
    'std_acc':  float(np.std([f['accuracy'] for f in all_fold_results])),
    'mean_f1':  float(np.mean([f['macro_f1'] for f in all_fold_results])),
    'std_f1':   float(np.std([f['macro_f1'] for f in all_fold_results])),
    'folds': all_fold_results,
}

with open(REPORT_DIR / 'M5_results.json', 'w') as f:
    json.dump(_ser(results), f, indent=2)
print(f'Saved → {REPORT_DIR / "M5_results.json"}')


In [ ]:
# ── Copy best.pt weights to /kaggle/working/ for easy download ────────────────
print('Weights available for download from /kaggle/working/:')
for fold_idx in range(5):
    src = CKPT_DIR / f'M5_fold{fold_idx}' / 'weights' / 'best.pt'
    dst = WORK_DIR / f'M5_fold{fold_idx}_best.pt'
    if src.exists():
        shutil.copy2(str(src), str(dst))
        sz = dst.stat().st_size / 1e6
        print(f'  ✓ M5_fold{fold_idx}_best.pt  ({sz:.1f} MB)')
    else:
        print(f'  ✗ Fold {fold_idx} — best.pt not found')

# Also copy results
shutil.copy2(str(REPORT_DIR / 'M5_results.json'), str(WORK_DIR / 'M5_results.json'))
print(f'  ✓ M5_results.json')
print(f'\nDownload from Output tab (right panel) → /kaggle/working/')
